# EEG_10b — MAML su GNN (Model-Agnostic Meta-Learning)

**Task**: Meta-learning con MAML per adattamento rapido cross-subject.
**Idea**: impara un'inizializzazione θ della GNN tale che con K gradient steps su un nuovo soggetto (support set) raggiunga buona accuracy sul query set.

**Inner loop**: 5 gradient steps su 20 trial del soggetto (support).
**Outer loop**: aggiorna θ per minimizzare la loss sul query set di tutti i soggetti del batch.

**Richiede**: `pip install learn2learn`


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os

CSV_ROOT      = "/home/daniele_u/data/raw_csv/training_set"
LABEL2IDX     = "/home/daniele_u/miralis-hypergraph-imagined-speech/configs/label_schemes/label2idx.json"
LABEL2CLUSTER = "/home/daniele_u/miralis-hypergraph-imagined-speech/configs/label_schemes/labelid2cluster_concr4.json"

SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Episodio
N_WAY      = 4
K_SHOT     = 5    # support per classe
N_QUERY    = 15   # query per classe (più ampio del ProtoNet per inner loop più stabile)

# Grafo
K_GRAPH    = 6

# GNN
HIDDEN_DIM = 64
N_LAYERS   = 3
DROPOUT    = 0.3

# MAML
INNER_LR    = 0.01
INNER_STEPS = 5       # gradient steps nell'inner loop
META_LR     = 1e-3
META_BATCH  = 4       # soggetti per meta-step
MAX_STEPS   = 2_000
FIRST_ORDER = True    # True = FOMAML (più veloce, quasi equivalente)

# W&B
WANDB_PROJECT = "miralis-imagined-speech"
WANDB_ENTITY  = "uras-daniele22-politecnico-di-milano"
RUN_NAME      = "eeg10b_MAML_GNN_concr4"


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import json, random, glob
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

try:
    import learn2learn as l2l
    print(f"learn2learn version: {l2l.__version__}")
except ImportError:
    raise ImportError("Installa learn2learn: pip install learn2learn")

import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Mapping label → cluster (identico a EEG_10a) ─────────────────────────────
with open(LABEL2IDX) as f:
    label2idx = json.load(f)

with open(LABEL2CLUSTER) as f:
    raw = json.load(f)
    label2cluster = {int(k): int(v) for k, v in raw.items()}

cluster2words = {c: [] for c in range(N_WAY)}
for word, idx in label2idx.items():
    c = label2cluster.get(idx)
    if c is not None:
        cluster2words[c].append(word)

print("Cluster sizes:", {c: len(w) for c, w in cluster2words.items()})


In [ ]:
# ── Trial loader e graph builder (identici a EEG_10a) ────────────────────────
def load_trial(csv_path: str):
    x = np.loadtxt(csv_path, delimiter=",", dtype=np.float32)
    if x.ndim == 1 or x.shape[0] < 61:
        return None
    mu  = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True) + 1e-6
    return (x - mu) / std


def pcc_edge_index(x_np: np.ndarray, k: int = K_GRAPH):
    N    = x_np.shape[0]
    corr = np.corrcoef(x_np)
    np.fill_diagonal(corr, -1.0)
    src, dst = [], []
    for i in range(N):
        for j in np.argsort(corr[i])[-k:]:
            src += [i, j]; dst += [j, i]
    return torch.tensor([src, dst], dtype=torch.long)


def arrays_to_batch(x_list):
    return Batch.from_data_list([
        Data(x=torch.from_numpy(x), edge_index=pcc_edge_index(x))
        for x in x_list
    ])


In [ ]:
# ── SubjectTaskSampler ────────────────────────────────────────────────────────
import glob

class SubjectTaskSampler:
    """
    Ogni 'task' MAML = un soggetto.
    sample_task(si) → (support_x, support_y, query_x, query_y)
    con support e query dallo stesso soggetto (MAML standard).
    """
    def __init__(self, subj_indices):
        self.index = {}
        subj_dirs  = sorted(glob.glob(f"{CSV_ROOT}/P*"))

        for si in subj_indices:
            if si >= len(subj_dirs):
                continue
            sdir = subj_dirs[si]
            self.index[si] = {c: [] for c in range(N_WAY)}
            for csv_path in glob.glob(f"{sdir}/**/*.csv", recursive=True):
                word    = Path(csv_path).stem.replace("_img","").replace("_conc","")
                idx     = label2idx.get(word)
                if idx is None: continue
                cluster = label2cluster.get(idx)
                if cluster is None: continue
                self.index[si][cluster].append(csv_path)

        min_needed = K_SHOT + N_QUERY
        self.valid  = [
            si for si, cd in self.index.items()
            if all(len(v) >= min_needed for v in cd.values())
        ]
        print(f"Task validi: {len(self.valid)} / {len(subj_indices)}")

    def sample_task(self, si=None):
        if si is None:
            si = random.choice(self.valid)
        sx, sy, qx, qy = [], [], [], []
        for c in range(N_WAY):
            paths = random.sample(self.index[si][c], K_SHOT + N_QUERY)
            for p in paths[:K_SHOT]:
                x = load_trial(p)
                if x is not None: sx.append(x); sy.append(c)
            for p in paths[K_SHOT:]:
                x = load_trial(p)
                if x is not None: qx.append(x); qy.append(c)
        return sx, sy, qx, qy


print("Building task samplers...")
train_sampler = SubjectTaskSampler(SUBJ_TRAIN)
val_sampler   = SubjectTaskSampler(SUBJ_VAL)
test_sampler  = SubjectTaskSampler(SUBJ_TEST)


In [ ]:
# ── GNN Classifier (con testa di classificazione per MAML) ───────────────────
class GNNClassifier(nn.Module):
    """
    GCN + global mean pool + MLP classificatore.
    MAML adatta TUTTI i parametri (encoder + head).
    """
    def __init__(self, in_ch=384, hidden=HIDDEN_DIM,
                 n_classes=N_WAY, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(GCNConv(in_ch, hidden))
        self.bns.append(nn.BatchNorm1d(hidden))
        for _ in range(n_layers - 1):
            self.convs.append(GCNConv(hidden, hidden))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.dropout   = dropout
        self.head      = nn.Linear(hidden, n_classes)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        h   = global_mean_pool(x, batch)        # (B, hidden)
        return self.head(h)                      # (B, n_classes)

    def forward_batch(self, batch_obj):
        return self.forward(batch_obj.x, batch_obj.edge_index, batch_obj.batch)


In [ ]:
# ── MAML setup con learn2learn ────────────────────────────────────────────────
base_model = GNNClassifier().to(device)

# learn2learn wrappa il modello: maml.clone() crea una copia con parametri
# indipendenti per l'inner loop
maml = l2l.algorithms.MAML(
    base_model,
    lr=INNER_LR,
    first_order=FIRST_ORDER,   # FOMAML se True
    allow_unused=True,
    allow_nograd=True,
)

meta_optimizer = torch.optim.Adam(maml.parameters(), lr=META_LR)
scheduler      = torch.optim.lr_scheduler.CosineAnnealingLR(
    meta_optimizer, T_max=MAX_STEPS
)

def inner_loop(learner, sx, sy, qx, qy):
    """
    Esegue INNER_STEPS sul support set, ritorna query loss + acc.
    learner = maml.clone() — copia con gradienti separati.
    """
    s_batch  = arrays_to_batch(sx).to(device)
    s_labels = torch.tensor(sy, dtype=torch.long, device=device)
    q_batch  = arrays_to_batch(qx).to(device)
    q_labels = torch.tensor(qy, dtype=torch.long, device=device)

    # Inner loop: adattamento al soggetto
    for _ in range(INNER_STEPS):
        logits     = learner.forward_batch(s_batch)
        inner_loss = F.cross_entropy(logits, s_labels)
        learner.adapt(inner_loss)          # un gradient step interno

    # Query loss (usata per outer loop)
    q_logits   = learner.forward_batch(q_batch)
    query_loss = F.cross_entropy(q_logits, q_labels)
    acc        = (q_logits.argmax(1) == q_labels).float().mean().item()
    return query_loss, acc


In [ ]:
# ── Meta-training loop ───────────────────────────────────────────────────────
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "notebook": "EEG_10b_maml_gnn",
        "n_way": N_WAY, "k_shot": K_SHOT, "n_query": N_QUERY,
        "inner_lr": INNER_LR, "inner_steps": INNER_STEPS,
        "meta_lr": META_LR, "meta_batch": META_BATCH,
        "max_steps": MAX_STEPS, "first_order": FIRST_ORDER,
        "hidden_dim": HIDDEN_DIM, "n_layers": N_LAYERS,
        "k_graph": K_GRAPH, "cluster_scheme": "concr4",
        "model": "MAML_GCN",
        "n_train_subj": len(train_sampler.valid),
    },
    reinit=True
)

best_val_acc = 0.0

for step in range(1, MAX_STEPS + 1):
    meta_optimizer.zero_grad()

    meta_loss_total, meta_acc_total = 0.0, 0.0

    for _ in range(META_BATCH):
        learner  = maml.clone()   # copia indipendente per questo task
        sx, sy, qx, qy = train_sampler.sample_task()
        if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
            continue

        q_loss, acc = inner_loop(learner, sx, sy, qx, qy)
        (q_loss / META_BATCH).backward()     # accumula gradiente outer
        meta_loss_total += q_loss.item()
        meta_acc_total  += acc

    meta_optimizer.step()
    scheduler.step()

    avg_loss = meta_loss_total / META_BATCH
    avg_acc  = meta_acc_total  / META_BATCH

    run.log({
        "train/meta_loss": avg_loss,
        "train/meta_acc":  avg_acc,
        "lr": scheduler.get_last_lr()[0],
        "step": step
    })

    # ── Validation ogni 100 step ──
    if step % 100 == 0:
        val_accs = []
        for _ in range(30):
            learner  = maml.clone()
            sx, sy, qx, qy = val_sampler.sample_task()
            if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
                continue
            with torch.no_grad():  # non aggiorna i meta-parametri
                # Nota: per MAML serve comunque il forward durante adapt.
                # Usiamo torch.enable_grad() solo nell'inner loop.
                pass
            # Val inner loop con enable_grad per adapt, ma no outer backward
            learner2 = maml.clone()
            sx2, sy2, qx2, qy2 = val_sampler.sample_task()
            if not sx2: continue
            s_b = arrays_to_batch(sx2).to(device)
            s_l = torch.tensor(sy2, dtype=torch.long, device=device)
            for _ in range(INNER_STEPS):
                logits = learner2.forward_batch(s_b)
                learner2.adapt(F.cross_entropy(logits, s_l))
            q_b = arrays_to_batch(qx2).to(device)
            q_l = torch.tensor(qy2, dtype=torch.long, device=device)
            with torch.no_grad():
                q_logits = learner2.forward_batch(q_b)
            acc = (q_logits.argmax(1) == q_l).float().mean().item()
            val_accs.append(acc)

        val_acc = float(np.mean(val_accs)) if val_accs else 0.0
        run.log({"val/acc": val_acc, "step": step})
        print(f"Step {step:4d} | loss {avg_loss:.4f} | train_acc {avg_acc:.3f} | val_acc {val_acc:.3f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(base_model.state_dict(), "/tmp/maml_best.pt")
            run.summary["best_val_acc"] = best_val_acc

print(f"\nMigliore val_acc: {best_val_acc:.3f} (chance: {1/N_WAY:.3f})")


In [ ]:
# ── Test finale: few-shot su soggetti mai visti ───────────────────────────────
base_model.load_state_dict(torch.load("/tmp/maml_best.pt"))
maml_test = l2l.algorithms.MAML(base_model, lr=INNER_LR,
                                  first_order=FIRST_ORDER,
                                  allow_unused=True, allow_nograd=True)

test_accs = []
for _ in range(200):
    learner = maml_test.clone()
    sx, sy, qx, qy = test_sampler.sample_task()
    if len(sx) < N_WAY * K_SHOT or len(qx) < N_WAY * N_QUERY:
        continue

    s_b = arrays_to_batch(sx).to(device)
    s_l = torch.tensor(sy, dtype=torch.long, device=device)

    for _ in range(INNER_STEPS):
        logits = learner.forward_batch(s_b)
        learner.adapt(F.cross_entropy(logits, s_l))

    q_b = arrays_to_batch(qx).to(device)
    q_l = torch.tensor(qy, dtype=torch.long, device=device)
    with torch.no_grad():
        q_logits = learner.forward_batch(q_b)
    test_accs.append((q_logits.argmax(1) == q_l).float().mean().item())

test_acc = float(np.mean(test_accs))
test_ci  = 1.96 * float(np.std(test_accs)) / np.sqrt(len(test_accs))

print(f"Test acc ({len(test_accs)} ep): {test_acc:.3f} ± {test_ci:.3f}")
print(f"Chance: {1/N_WAY:.3f}")

run.summary["test_acc"]   = test_acc
run.summary["test_ci_95"] = test_ci
run.finish()
